In [ ]:
import pandas as pd
from pyreadstat import read_sav
import re 
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.compose import ColumnTransformer
import copy

def transform_with_captured_group(text, capturing_group:str=None, prefix:str="", suffix:str=""):
    """Function that takes a text and using a regex capturing group adjust the text to keep only the captured group and optionally adding a prefix and/or suffix to the string."""
    return f"{prefix}{re.search(capturing_group, text).group(1)}{suffix}"

def same_answer_quality_check(dataframe_to_update:pd.DataFrame, column_selection_like:str="", column_selection_regex:str="") -> pd.DataFrame:
    """
    Function that checks strightlining behaviour or same anwer patterns for a series of variables. Useful to detecting quality issues with the data set used in segmentations. 
    
    :param dataframe_to_update: Original Data Frame
    :type dataframe_to_update: pd.DataFrame
    :param column_selection_like: Selecting the columns to analyze with a pd.DataFrame.filter(like="Pattern") function. 
    :type column_selection_like: str
    :param column_selection_regex: Selecting the columns to analyze with a pd.DataFrame.filter(regex=r"Pattern") function. 
    :type column_selection_regex: str
    :return: Dataframe with the added columns of data quality. Uses the filter "like" or "regex" patter as the column name with "_QUALITY_SCORE" suffix as the name. 
    :rtype: DataFrame
    """
    
    updated_frame = dataframe_to_update.copy()

    if column_selection_like != "":
        mode_series = updated_frame.filter(like=column_selection_like).mode(axis=1)[0]
        equal_counts_series = updated_frame.filter(like=column_selection_like).eq(mode_series, axis="index").sum(axis=1)
        total_counts_series = updated_frame.filter(like=column_selection_like).notna().sum(axis=1)
        updated_frame[f"{column_selection_like}_QUALITY_SCORE"] = equal_counts_series.div(total_counts_series)

    elif column_selection_regex != "":
        mode_series = updated_frame.filter(regex=column_selection_regex).mode(axis=1)[0]
        equal_counts_series = updated_frame.filter(regex=column_selection_regex).eq(mode_series, axis="index").sum(axis=1)
        total_counts_series = updated_frame.filter(regex=column_selection_regex).notna().sum(axis=1)
        updated_frame[f"{column_selection_regex}_QUALITY_SCORE"] = equal_counts_series.div(total_counts_series)
    else:
        print("ERROR: Please supply a filter valid filter string, either as like filter or regex filter.")
    
    return updated_frame

def rename_meta_variables(meta, rename_map: dict, inplace: bool = False):
    """
    Return a metadata_container whose variable-name-keyed structures use the new names.

    Parameters
    ----------
    meta : pyreadstat metadata_container
    rename_map : dict  {old_variable_name: new_variable_name}
    inplace : bool     modify the given meta instead of a copy
    """
    new_meta = meta if inplace else copy.deepcopy(meta)

    def _rn(name):
        return rename_map.get(name, name)

    # 1) attributes that are dicts keyed by variable name
    dict_attrs = [
        "column_names_to_labels",
        "variable_value_labels",
        "variable_to_label",
        "variable_measure",
        "variable_display_width",
        "variable_alignment",
        "variable_storage_width",
        "original_variable_types",
        "readstat_variable_types",
        "missing_ranges",
        "missing_user_values",
        "variable_format",
    ]
    for attr in dict_attrs:
        old = getattr(new_meta, attr, None)
        if isinstance(old, dict):
            setattr(new_meta, attr, {_rn(k): v for k, v in old.items()})

    # 2) attributes that are lists of variable names
    list_attrs = ["column_names", "mr_sets"]
    for attr in list_attrs:
        old = getattr(new_meta, attr, None)
        if isinstance(old, list):
            setattr(new_meta, attr, [_rn(v) for v in old])

    # 3) multiple response sets: dict of set_name -> {..., "variable_list": [...]}
    mrsets = getattr(new_meta, "mrset", None) or getattr(new_meta, "multiple_response_sets", None)
    if isinstance(mrsets, dict):
        for set_name, cfg in mrsets.items():
            if isinstance(cfg, dict) and isinstance(cfg.get("variable_list"), list):
                cfg["variable_list"] = [_rn(v) for v in cfg["variable_list"]]

    return new_meta

def dataframe_stacking(df:pd.DataFrame, id_vars_list:list, key_pattern:str, keep_key:bool=True, key_column_name:str="STACK_KEY") -> pd.DataFrame:
    """
    Reshape a DataFrame from wide format to stacked format.

    All columns not listed in ``id_vars_list`` are treated as value columns.
    Their names are transformed using ``key_pattern``. If ``keep_key`` is
    ``True``, the full regex match is stored in a key column; the matching
    text is removed from the column names in either case. Rows with missing
    values are dropped before the result is pivoted back to wide format.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame to reshape.
    id_vars_list : list of str
        Columns to retain as identifier variables.
    key_pattern : str
        Regular expression matched against value-column names. The full match
        is used as the key when ``keep_key`` is ``True``.
    keep_key : bool, default=True
        Whether to include the extracted key as an identifier column.
    key_column_name : str, default="STACK_KEY"
        Name of the key column when ``keep_key`` is ``True``.

    Returns
    -------
    pandas.DataFrame
        Reshaped DataFrame with the identifier columns and pivoted values.
    """
    
    stacked_df = df.copy()
    stacked_df = stacked_df.melt(id_vars=id_vars_list)    
    extended_id_vars_list = id_vars_list.copy()
    extended_id_vars_list.append(key_column_name)

    if keep_key:
        stacked_df[key_column_name] = stacked_df['variable'].apply(lambda var: re.match(pattern=key_pattern, string=var)[0])
        
    stacked_df["variable"] = stacked_df["variable"].apply(lambda var: re.sub(pattern=key_pattern, repl="", string=var))
    stacked_df.dropna(subset=["value"], inplace=True)
    
    if keep_key:
        stacked_df = stacked_df.pivot(columns="variable", values="value", index=extended_id_vars_list)
    else: 
        stacked_df = stacked_df.pivot(columns="variable", values="value", index=id_vars_list)
    
    return stacked_df.reset_index()

class ColumnMapper(BaseEstimator, TransformerMixin):

    def __init__(self, mappings):
        self.mappings = mappings

    def fit(self, X, y=None):
        self.feature_names_in_ = list(X.columns)
        return self

    def transform(self, X):
        X = X.copy()

        for col, mapping in self.mappings.items():
            if col in X.columns:
                X[col] = X[col].replace(mapping)

        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_
    
class TopNBoxEncoder(BaseEstimator, TransformerMixin):
    """
    Create cumulative Top-N-Box binary columns.

    Parameters
    ----------
    top_n_boxes : dict
        Dictionary where:
            key   = column name
            value = maximum number of Top-N-Box columns to create

        Example:
            {
                "A": 2,
                "B": 3
            }

        This creates:
            A_T1B, A_T2B
            B_T1B, B_T2B, B_T3B

    max_values : dict or None, default=None
        Optional dictionary containing the theoretical maximum for each
        column.

        Example:
            {
                "A": 5,
                "B": 7
            }

        If a maximum is not provided, it is learned from the training data.

    handle_missing : {"zero", "nan"}, default="zero"
        Determines how missing values are encoded:

        - "zero": missing values produce 0 in all Top-N-Box columns.
        - "nan": missing values remain NaN in all generated columns.

    keep_original : bool, default=False
        If True, return the original columns in addition to the generated
        Top-N-Box columns.

        Usually False is preferable when used inside a ColumnTransformer.
    """

    def __init__(
        self,
        top_n_boxes,
        max_values=None,
        handle_missing="zero",
        keep_original=False,
    ):
        self.top_n_boxes = top_n_boxes
        self.max_values = max_values
        self.handle_missing = handle_missing
        self.keep_original = keep_original

    def fit(self, X, y=None):
        X = self._to_dataframe(X)

        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.n_features_in_ = X.shape[1]

        self._validate_parameters(X)

        # Store the maximum used for each column.
        self.max_values_ = {}

        supplied_max_values = (
            {} if self.max_values is None else self.max_values
        )

        for column in self.top_n_boxes:
            if column in supplied_max_values:
                maximum = supplied_max_values[column]
            else:
                maximum = X[column].max(skipna=True)

            if pd.isna(maximum):
                raise ValueError(
                    f"Cannot determine the maximum for column {column!r}. "
                    "The column contains only missing values. Provide its "
                    "maximum through max_values."
                )

            self.max_values_[column] = maximum

        self.output_feature_names_ = self._build_feature_names()

        return self

    def transform(self, X):
        check_is_fitted(
            self,
            attributes=[
                "feature_names_in_",
                "max_values_",
                "output_feature_names_",
            ],
        )

        X = self._to_dataframe(
            X,
            expected_columns=self.feature_names_in_,
        )

        generated_columns = {}

        if self.keep_original:
            for column in X.columns:
                generated_columns[column] = X[column].copy()

        for column, n_boxes in self.top_n_boxes.items():
            values = pd.to_numeric(X[column], errors="coerce")
            maximum = self.max_values_[column]

            for n in range(1, n_boxes + 1):
                threshold = maximum - n + 1
                output_name = f"{column}_T{n}B"

                encoded = values.ge(threshold)

                if self.handle_missing == "nan":
                    encoded = encoded.astype("Float64")
                    encoded = encoded.mask(values.isna(), pd.NA)
                else:
                    encoded = encoded.fillna(False).astype(np.uint8)

                generated_columns[output_name] = encoded

        return pd.DataFrame(
            generated_columns,
            index=X.index,
        )

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, attributes=["output_feature_names_"])
        return np.asarray(self.output_feature_names_, dtype=object)

    def _build_feature_names(self):
        names = []

        if self.keep_original:
            names.extend(self.feature_names_in_)

        for column, n_boxes in self.top_n_boxes.items():
            names.extend(
                f"{column}_T{n}B"
                for n in range(1, n_boxes + 1)
            )

        return np.asarray(names, dtype=object)

    def _validate_parameters(self, X):
        if not isinstance(self.top_n_boxes, dict):
            raise TypeError("top_n_boxes must be a dictionary.")

        if self.max_values is not None:
            if not isinstance(self.max_values, dict):
                raise TypeError("max_values must be a dictionary or None.")

        if self.handle_missing not in {"zero", "nan"}:
            raise ValueError(
                "handle_missing must be either 'zero' or 'nan'."
            )

        missing_columns = set(self.top_n_boxes) - set(X.columns)

        if missing_columns:
            raise ValueError(
                "The following columns were not found in X: "
                f"{sorted(missing_columns)}"
            )

        invalid_n_values = {
            column: n
            for column, n in self.top_n_boxes.items()
            if (
                not isinstance(n, (int, np.integer))
                or isinstance(n, bool)
                or n < 1
            )
        }

        if invalid_n_values:
            raise ValueError(
                "The number of Top-N-Box variables must be a positive "
                f"integer. Invalid values: {invalid_n_values}"
            )

        supplied_max_values = (
            {} if self.max_values is None else self.max_values
        )

        invalid_boxes = {
            column: n
            for column, n in self.top_n_boxes.items()
            if (
                column in supplied_max_values
                and n > supplied_max_values[column]
            )
        }

        if invalid_boxes:
            raise ValueError(
                "The requested number of boxes cannot be greater than "
                "the supplied maximum value. Invalid values: "
                f"{invalid_boxes}"
            )

    @staticmethod
    def _to_dataframe(X, expected_columns=None):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()

            if expected_columns is not None:
                missing = set(expected_columns) - set(X_df.columns)

                if missing:
                    raise ValueError(
                        f"Missing columns during transform: {sorted(missing)}"
                    )

                X_df = X_df.loc[:, expected_columns]

            return X_df

        X_array = np.asarray(X)

        if X_array.ndim != 2:
            raise ValueError("X must be a two-dimensional structure.")

        if expected_columns is None:
            columns = [
                f"x{i}" for i in range(X_array.shape[1])
            ]
        else:
            if X_array.shape[1] != len(expected_columns):
                raise ValueError(
                    "The number of columns during transform does not match "
                    "the number seen during fit."
                )

            columns = expected_columns

        return pd.DataFrame(X_array, columns=columns)    

# Preanalisis 

The preanalisis for a segmentation focuses on the quality of the responses we have for the variables we are going to use to segment as well as confirming we have answers for all the variables we are going to use as the core of our segmentation process (no NA's). 

The main quality control is confirm how much of the sample have the same answers (identcial) in all attributes in key batteries, the function returns the percentage of identical answers. We normally work with records that have < 80% of identical answers. 

In [ ]:
# Read SPSS
data_preanalysis, meta_preanalysis = read_sav("MastelloneMadres.sav")

In [ ]:
# Obtain the describe DF of the database (we should look for the number of answers in the key variables we are going to use to segment)
data_describe = data_preanalysis.describe(include="all").T

data_preanalysis.set_index(["Respondent_Serial"], inplace=True)

In [ ]:
### FIXING VARIABLES NAMES FOR THE ANALYSIS AND OTHER DATABASE ISSUE - PROJECT DEPENDENT 

# FUNC
necesidades_funcionales = data_preanalysis.filter(regex=r"NECESIDA\w+\d_\w+FUNCIONAL\w+\d_\d+")
necesidades_funcionales = necesidades_funcionales.melt(ignore_index=False)
necesidades_funcionales["adjusted_variable"] = necesidades_funcionales["variable"].apply(
    lambda text: transform_with_captured_group(
        text, 
        capturing_group=r"(\d+)_*slic", 
        prefix=transform_with_captured_group(
            text,
            capturing_group=r"(\d+)_\d+",
            prefix="LOOP_",
            suffix="_NECESIDADES_FUNCIONALES_SLICE_"
        )))
necesidades_funcionales.dropna(inplace=True)
necesidades_funcionales = necesidades_funcionales.pivot(columns="adjusted_variable", values="value")


# EMO
necesidades_emocionales = data_preanalysis.filter(regex=r"NECESIDA\w+\d_\w+EMO\w+\d_\d+")
necesidades_emocionales = necesidades_emocionales.melt(ignore_index=False)
necesidades_emocionales["adjusted_variable"] = necesidades_emocionales["variable"].apply(
    lambda text: transform_with_captured_group(
        text, 
        capturing_group=r"(\d+)_*slic", 
        prefix=transform_with_captured_group(
            text,
            capturing_group=r"(\d+)_\d+",
            prefix="LOOP_",
            suffix="_NECESIDADES_EMOCIONALES_SLICE_"
        )))
necesidades_emocionales.dropna(inplace=True)
necesidades_emocionales = necesidades_emocionales.pivot(columns="adjusted_variable", values="value")

# Join with data 
data_preanalysis = pd.merge(
    left=data_preanalysis,
    right=necesidades_funcionales,
    left_index=True,
    right_index=True,
    how="left"
)

data_preanalysis = pd.merge(
    left=data_preanalysis,
    right=necesidades_emocionales,
    left_index=True,
    right_index=True,
    how="left"
)

In [ ]:
# Calculate for each relevant grid the 
for i_grid in ["FUNCIONALES","EMOCIONALES"]:
    for i_loop in [1,2]:
        same_answer_quality_check(data_preanalysis, f'LOOP_{i_loop}_NECESIDADES_{i_grid}')
        

In [ ]:
# Excel output
with pd.ExcelWriter("preanalisis.xlsx", engine="openpyxl") as writer:
    #data.to_excel(writer, sheet_name="data", index=False)
    data_describe.to_excel(writer, sheet_name="describe_data")
    data_preanalysis.filter(like="_QUALITY_SCORE").to_excel(writer, sheet_name="quality")

# Data Cleaning

In [ ]:
# Loads the data cleaning configuration file. We have to prepare an Excel with the following columns: 
# VAR = Original variable name, only keep those we are going to use
# N_VAR = New variable name. Copy original ones and update those needed for preprocessing reasons. 
# TYPE = (Optional) Used for stacking the dataframe. We identify ID vars vs STACKING vars, we should add labels indicating their role in the stacking process. 

VAR_CLEANING_FRAME = pd.read_excel("Data Preparation and Analysis Plan.xlsx", sheet_name="VAR_CLEANING_FRAME")

VAR_CLEANING_FRAME.set_index("VAR", inplace=True)
MAIN_VAR_DICT = VAR_CLEANING_FRAME["N_VAR"].to_dict()


In [ ]:
# Loads, select variables and renames relevant columns

data_cleaning, meta_cleaning = read_sav("MastelloneMadres.sav")
data_cleaning = data_cleaning[VAR_CLEANING_FRAME.index] # Selects only relevant variables 
meta_cleaning = rename_meta_variables(meta_cleaning, MAIN_VAR_DICT) # Renames meta
data_cleaning = data_cleaning.rename(columns=MAIN_VAR_DICT) # Renames column names 

In [ ]:
# Reads stacked configuration 
ID_VAR_LIST = VAR_CLEANING_FRAME[VAR_CLEANING_FRAME["TYPE"]=="ID"].index.to_list()

# Applies stacking to the dataset 
data_cleaning = dataframe_stacking(data_cleaning, ID_VAR_LIST, r"(Ocasion\d)", keep_key=True)

# ONLY FOR MASTELLONE - ANOTHER ROUND OF STACKING TO CONSOLIDATE LOOSE VARIABLES 
ID_VAR_LIST_NEW = ID_VAR_LIST.copy()
ID_VAR_LIST_NEW.append("STACK_KEY")

data_cleaning = dataframe_stacking(data_cleaning, ID_VAR_LIST_NEW, key_pattern=r"(Block(?:_Grupo)?_?\d)", keep_key=False)

In [7]:
data_cleaning.columns.to_series().to_clipboard()

In [8]:
PIPE_CONFIG_FRAME = pd.read_excel("Data Preparation and Analysis Plan.xlsx", sheet_name="PIPE_CONFIG_FRAME")

In [9]:
PIPE_CONFIG_FRAME.set_index("VAR", inplace=True)


DICT_REP_COLS = PIPE_CONFIG_FRAME.dropna(subset="DICT_REP").index.to_list()
ONE_HOT_COLS = PIPE_CONFIG_FRAME.dropna(subset="ONE_HOT").index.to_list()
TOP_N_COLS = PIPE_CONFIG_FRAME.dropna(subset="TOP_N").index.to_list()

import ast

DICT_REP_MAPPING = PIPE_CONFIG_FRAME["DICT_REP"].dropna().apply(lambda x: ast.literal_eval(x)).to_dict()
TOP_N_COLS_MAPPING = PIPE_CONFIG_FRAME["TOP_N"].dropna().astype("int").to_dict()

In [10]:
preprocessing_trasnformer = ColumnTransformer([
    ("DRT", ColumnMapper(mappings=DICT_REP_MAPPING), DICT_REP_COLS),
    ("OHT", OneHotEncoder(sparse_output=False), ONE_HOT_COLS),
    ("TBT", TopNBoxEncoder(top_n_boxes=TOP_N_COLS_MAPPING, handle_missing="zero", keep_original=True), TOP_N_COLS)
], remainder="passthrough")

preprocessing_trasnformer.fit(data_cleaning)

clean_data = pd.DataFrame(data=preprocessing_trasnformer.transform(data_cleaning), columns=preprocessing_trasnformer.get_feature_names_out())

clean_data = same_answer_quality_check(clean_data, column_selection_regex=r"EMOCIONALES_Slice_\d*$")
clean_data = same_answer_quality_check(clean_data, column_selection_regex=r"FUNCIONALES_Slice_\d*$")

clean_data.to_csv("clean_data.csv")
clean_data.to_pickle("clean_data.plk")

In [11]:
clean_data = pd.read_pickle("clean_data.plk")

In [12]:
clean_data.columns.to_series().to_clipboard()